In [91]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
np.set_printoptions(threshold=np.inf)

## Read in files and establish standardized dates and compare times

Combine all fiscal years and establish date fields on the nationwide database before selecting for North Carolina arrests. 

In [92]:
df26 = pd.read_excel('/Users/cmcglade/Documents/Projects/ice-update-april-2026/2026-ICLI-00005_Arrests_FY26_20260311_Redacted.xlsx', skiprows=6)
df25 = pd.read_excel('/Users/cmcglade/Documents/Projects/ice-update-april-2026/2026-ICLI-00005_Arrests_FY25_20260311_Redacted.xlsx', skiprows=6)
df24 = pd.read_excel('/Users/cmcglade/Documents/Projects/ice-update-april-2026/2026-ICLI-00005_Arrests_FY24_20260311_Redacted.xlsx', skiprows=6)
df23 = pd.read_excel('/Users/cmcglade/Documents/Projects/ice-update-april-2026/2026-ICLI-00005_Arrests_FY23_20260311_Redacted.xlsx', skiprows=6)

df = pd.concat([df26,df25,df24,df23])

#create a year column
df['year'] = df['Apprehension Date'].dt.year
df['year'] = df['year'].astype(str)

#create a month column
df['month'] = df['Apprehension Date'].dt.month
df['month'] = df['month'].astype(str)

#create a generic day column so I can group by month-year
df['day'] = '01'

#create a standardized date for month-year
df['standardized_date'] = df['year']+'-'+df['month']+'-'+df['day']
df['standardized_date'] = pd.to_datetime(df['standardized_date'])


df['comparetime'] = np.where(
    df['Apprehension Date'].dt.normalize().between('2026-01-20', '2026-03-10'),
    'this_year',
    np.where(
        df['Apprehension Date'].dt.normalize().between('2025-01-20', '2025-03-10'),
        'last_year',
        'not in compare range'
    )
)

## Break the file into two pieces

One file is missing state values and the other has them.

Filter for NC and set aside the the clean file for later, move the messy file on to the next step.

In [93]:
# Lowercase columns first
df['Apprehension Site Landmark'] = df['Apprehension Site Landmark'].str.lower()
df['Apprehension State'] = df['State'].str.lower()
df['Apprehension AOR'] = df['TOA Current Duty AOR'].str.lower()

#create two separate files, one where there is no state and the other where there is. I will clean the no_state file.
no_state = df[df['Apprehension State'].isnull()].copy()
print('No State: ', no_state.shape)


state = df[df['Apprehension State'].notna()].copy()
print('State: ', state.shape)

#filter for NC
nc1 = state[state['Apprehension State']== 'north carolina']

No State:  (107376, 32)
State:  (606088, 32)


## Step Two: Peel states out of the site landmark column

The site landmark column has a variety of location types. Sometimes, the state is referenced with a comma and the two letter abbreviation. Here, I specify valid state abbreviations and run a regex argument to pick up any of those and dump them into a new column called 'extracted state.'

I then create a new file that grabs all rows where this was successful, filtered for nc and set that aside. I also create a file where there was not a state abbreviation within the landmark column, so the extracted state column is blank.

That file, which I called no_state2 moves on to the next step.

In [94]:
# List of valid US state abbreviations
us_states = [
    'al','ak','az','ar','ca','co','ct','de','fl','ga','hi','id','il','in','ia','ks','ky','la',
    'me','md','ma','mi','mn','ms','mo','mt','ne','nv','nh','nj','nm','ny','nc','nd','oh','ok',
    'or','pa','ri','sc','sd','tn','tx','ut','vt','va','wa','wv','wi','wy'
]

# Extract potential 2-letter state code from "Apprehension Site Landmark"
no_state['extracted_state'] = (
    no_state['Apprehension Site Landmark']
      .str.extract(r',\s*([a-zA-Z]{2})(?![a-zA-Z])', expand=False)  # match exactly two letters after a comma
      .str.lower()                                                  # make them lowercase
      .where(lambda x: x.isin(us_states))                           # only keep valid US abbreviations
)
#create a dataframe that has state names now that didn't before
extractedstate = no_state[no_state['extracted_state'].notna()].copy()
print('Rows with extracted states from Site Landmark: ', extractedstate.shape)

#create a dataframe that is still missing a state
no_state2 = no_state[no_state['extracted_state'].isnull()].copy()
print('Rows where there is no Apprehension State and no clear state abbreviation in landmark: ', no_state2.shape)

extractedstate['Apprehension State'] = extractedstate['extracted_state']
extractedstate = extractedstate.drop(columns=['extracted_state'])
#extractedstate['Apprehension State'].unique()

#filter for nc
nc2 = extractedstate[extractedstate['Apprehension State'] == 'nc']

Rows with extracted states from Site Landmark:  (10337, 33)
Rows where there is no Apprehension State and no clear state abbreviation in landmark:  (97039, 33)


## Step Three: Search for North Carolina counties

Using the file where there was no state abbreviation found, and no state referenced in the Apprehension State column, I create a new file that grabs any rows where the Area of Responsibility is Atlanta.

[In a separate file](https://colab.research.google.com/drive/1644oPILC8rSwN2LFSWIYkOxGI0vIG85B?usp=sharing), I used the tigerline census shapefile to pull all the county names in North Carolina, Georgia and South Carolina (our AOR) and then ran a groupby to identify counties in North Carolina that share a name with counties in the other two states. I excluded those, and created a list of counties that only exist in North Carolina.

From here, I created a new file called nc_county_names that grabs any row where the counties in my list are mentioned. I set that aside for later.

I then also created a new file where those counties were not mentioned, and printed out the unique landmark variables and David Raynor looked at them and picked out several that were definitly in North Carolina, that I would not know about because I'm new here. I created a list of them, and then created a file that pulled those rows out called nc_stragglers.

I had also created a file where there was no Apprehension AOR listed. I looked at the unique variables there and found one location that was definitely in North Carolina. Raynor found some that could possibly be, but we couldn't be sure, so I excluded them.

The drawback of this is that I am likely missing people who were picked up in North Carolina, but simply because their county shares a name with another one in neighboring states, they are excluded. Or, in some places where the AOR is missing and the landmark doesn't provide obvious clues. There isn't really anything I can do about this, though, as there are not any other columns that would provide clues to confirm that the person was in North Carolina and not Georgia, or South Carolina.

At the very end of all of this, I stack my three dataframes into one: the file that found North Carolina county names, the file that found places that are in North Carolina and are in the AOR, and the file that found a few rows that were in North Carolina but were missing an AOR.

In [95]:
#create a dataframe that might contain NC arrests
potential_nc = no_state2[no_state2['Apprehension AOR'].str.contains('atlanta', na=False)]

no_aor = no_state2[no_state2['Apprehension AOR'].isnull()]

print('Rows where AOR is Atlanta', potential_nc.shape)
print('Rows where AOR is NULL', no_aor.shape)

nc_counties = [
'alamance', 'alexander', 'alleghany', 'anson', 'ashe', 'avery',
'bertie', 'bladen', 'brunswick', 'buncombe', 'cabarrus',
'caldwell', 'carteret', 'caswell', 'catawba', 'chowan', 'cleveland',
'columbus', 'craven', 'cumberland', 'currituck', 'dare', 'davidson',
'davie', 'duplin', 'durham', 'edgecombe', 'gaston', 'gates', 'graham',
'granville', 'guilford', 'halifax', 'harnett', 'haywood', 'henderson', 'hertford',
'hoke', 'hyde', 'iredell', 'johnston', 'lenoir', 'martin', 'mcdowell', 'mecklenburg',
'moore', 'nash', 'new hanover', 'northampton', 'onslow', 'orange', 'pamlico',
'pasquotank', 'pender', 'perquimans', 'person', 'pitt', 'robeson',
'rockingham', 'rowan', 'rutherford', 'sampson', 'scotland', 'stanly', 'stokes',
'surry', 'swain', 'transylvania', 'tyrrell', 'vance', 'wake',
'watauga', 'wilson', 'yadkin', 'yancey'

]

import re

nc_county_names = potential_nc[
    potential_nc['Apprehension Site Landmark']
    .str.lower()
    .str.contains('|'.join([re.escape(c) for c in nc_counties]), na=False)
]

import re

other_potential_nc = potential_nc[
    ~potential_nc['Apprehension Site Landmark']
    .str.lower()
    .str.contains('|'.join([re.escape(c) for c in nc_counties]), na=False)
]


print('Rows in NC counties and are also in the Atlanta AOR (dupe name counties excluded: )', nc_county_names.shape)
print('Rows that are in the Atlanta AOR but do not have any NC county names', other_potential_nc.shape)

defnc = [
    'butner bop facility',
    'charlotte general area',
    'nc department of corrections',
    'rdu general area, non-specific',
    'ero charlotte',
    'edgecomb county jail',
    'butner medical facility',
    'butner medical facility',
    'nc dept of corrections facility',
    'tabor doc'
]



nc_stragglers = other_potential_nc[
    other_potential_nc['Apprehension Site Landmark'].str.lower().isin(defnc)
]

nc_stragglers2 = no_aor[
    no_aor['Apprehension Site Landmark'].isin([
        'sc. nc line from mecklenburg co e to richmond co; n to nc/va line',
        'ero charlotte'
    ])
]

print('NC locations in Atlanta AOR that did not get picked up by county name: ', nc_stragglers.shape)
print('Rows where AOR is Atlanta but there is no landmark info: ', potential_nc[potential_nc['Apprehension Site Landmark'].isnull()].shape)

#no_aor['Apprehension Site Landmark'].unique()

nc3 = pd.concat([nc_stragglers, nc_county_names])


print('NC locations in Atlanta AOR that did not get picked up by county name: ', nc_stragglers.shape)
print('NC locations that had no AOR at all: ', nc_stragglers2.shape)
print('Rows where AOR is Atlanta but there is no landmark info: ', potential_nc[potential_nc['Apprehension Site Landmark'].isnull()].shape)



nc3 = pd.concat([nc_stragglers, nc_stragglers2, nc_county_names])
#other_potential_nc['Apprehension Site Landmark'].unique()


Rows where AOR is Atlanta (1447, 33)
Rows where AOR is NULL (5565, 33)
Rows in NC counties and are also in the Atlanta AOR (dupe name counties excluded: ) (142, 33)
Rows that are in the Atlanta AOR but do not have any NC county names (1305, 33)
NC locations in Atlanta AOR that did not get picked up by county name:  (91, 33)
Rows where AOR is Atlanta but there is no landmark info:  (435, 33)
NC locations in Atlanta AOR that did not get picked up by county name:  (91, 33)
NC locations that had no AOR at all:  (19, 33)
Rows where AOR is Atlanta but there is no landmark info:  (435, 33)


## Step Four: Putting it all together

The first file I created (nc1) contained rows that all included the state name. The second file I created (nc2) has a state name as well, extracted from the landmark column.

I filtered for North Carolina in both of these.

The third file I created (nc3) combines rows where AOR is missing, but the landmark site had obvious ties to NC, AOR is Atlanta and the landmark had a NC-specific county name, AOR is Atlanta and the landmark had a place that was in North Carolina, identified by Raynor.

I stack these files on top of each other to make a master file called nc.

I have to do a little cleaning at this step, too.

There are duplicate rows, so I drop those. And then, I need to drop the rows where ICE clearly made a mistake when they listed the arrest in North Carolina, because they have the area of responsibility in other places and the arrest landmark also in other places.

The Deportation Data Project suggested people also dedupe rows where the apprehension date and the unique ID is repeated. I went a little further, noticing that some people have two arrests in one day, and decided to get rid of the second instance where someone was arrested twice in one day.

I merge the groupby back to the nc file and then run a command to keep only the first iteration of each unique ID.

You end up with 10,984 arrests.

In [96]:
nc = pd.concat([nc1,nc2,nc3])
nc = nc.drop_duplicates()
nc = nc[
    (nc['Apprehension AOR'] == 'atlanta area of responsibility') |
    (nc['Apprehension AOR'].isnull())
]
print(nc.shape)

(10984, 33)


## Establish date parameters

I want to track the arrests that happened on holidays and during Charlotte's web, so this code establishes flags for each of those. 

In [97]:
nc['Apprehension Date'] = nc['Apprehension Date'].dt.date
nc['Apprehension Date'] = pd.to_datetime(nc['Apprehension Date']).dt.normalize()
nc['Apprehension Date'] = pd.to_datetime(nc['Apprehension Date'])

nc['clt_web'] = np.where(
    nc['Apprehension Date'].between('2025-11-15', '2025-11-19'),
    'yes',
    'no'
)

nc['xmaseve'] = np.where(nc['Apprehension Date'] == '2025-12-24', 'yes', 'no')
nc['xmas']    = np.where(nc['Apprehension Date'] == '2025-12-25', 'yes', 'no')
nc['nye']     = np.where(nc['Apprehension Date'] == '2025-12-31', 'yes', 'no')
nc['nyd']     = np.where(nc['Apprehension Date'] == '2026-01-01', 'yes', 'no')
nc['thx']     = np.where(nc['Apprehension Date'] == '2025-12-27', 'yes', 'no')

## Identify duplicates

To identify potential duplicates, I am going to find people who are arrested on the same day, with the same ID. Group by these two fields, and then merge it back to the main database.

In [98]:
gb = nc.groupby(['Anonymized Identifier', 'Apprehension Date']).size().to_frame(name='records').reset_index()
gb['app-date'] = gb['Apprehension Date'].astype(str)
gb['unique_id_date'] = gb['Anonymized Identifier']+gb['app-date']
nc = pd.merge(nc, gb,how='left')
print('Check count after merge: ', nc.shape)

Check count after merge:  (10984, 42)


Separate the dupes from the non-dupes, and the ones that don't have unique IDs. In the dupes file, keep only the first row for each unique id and date. In the null_id file, keep only the first row for each date and landmark. Then put the nondupes, the dupes back and the null_id together.

In [99]:
dupes = nc[nc['records']>1]
print('Total rows that are likely duplicates: ', dupes.shape)
nondupes = nc[nc['records'] == 1]
null_id = nc[nc['Anonymized Identifier'].isnull()]
print('Total rows with no ID: ', null_id.shape)
print('Total rows that are not duplicates: ', nondupes.shape)
print('Dupes+noID+nondupes: ', 157+37+10790)
dupes = dupes.drop_duplicates(subset='unique_id_date', keep='first')
print('Unique arrests left from dupes file: ', dupes.shape)
null_id = null_id.drop_duplicates(subset=['Apprehension Date', 'Apprehension Site Landmark'], keep='first')
print('Unique arrests left from null file: ', null_id.shape)
nc_final = pd.concat([dupes,nondupes,null_id])
print('Total arrests, deduped: ', nc_final.shape)

Total rows that are likely duplicates:  (157, 42)
Total rows with no ID:  (37, 42)
Total rows that are not duplicates:  (10790, 42)
Dupes+noID+nondupes:  10984
Unique arrests left from dupes file:  (78, 42)
Unique arrests left from null file:  (10, 42)
Total arrests, deduped:  (10878, 42)


In [100]:
print('Total rows removed from likely dupes: ', 157-78)
print('Total rows removed from missing IDs: ', 37-10)

Total rows removed from likely dupes:  79
Total rows removed from missing IDs:  27


In [101]:
print('Sanity Check', 10984-79-27)

Sanity Check 10878


In [102]:
nc_final.to_csv('/Users/cmcglade/Documents/Projects/ice-update-april-2026/processed-data/nc-arrests-april-26-update.csv', index=False)